# 04 — Baseline (Logistic Regression)

**Workstream**: Modeling — baseline  ·  **Owner**: Bella (backup: Deepak)  ·  **Last touched**: 2026-06-02

**What this notebook decides**

Train and evaluate the **baseline** — a class-balanced logistic regression
on the contract feature matrix — and produce the first reportable model
performance numbers for this project. This is the floor we measure XGBoost
(Phase 5) against.

**What baseline gets us**

  1. **A defensibility floor.** Calibrated logreg with leak-free features
     can stand on its own in a research write-up. If XGBoost only barely
     beats this, we ship this.
  2. **A pipeline smoke test.** A baseline that crashes is more informative
     than a fancy model that crashes. Every feature pre-processor and
     every column dtype gets exercised here first.
  3. **A coefficient-level look at what the features are saying.** Useful
     for the model card even if it's not the final estimator.

**Why this notebook is short** — same reason as the previous two. The
feature contract and the pipeline factory live in
[`src/foodsafety/models/baseline.py`](../src/foodsafety/models/baseline.py).
Eval metrics live in [`src/foodsafety/models/evaluate.py`](../src/foodsafety/models/evaluate.py).
The temporal splitter lives in [`src/foodsafety/utils/time.py`](../src/foodsafety/utils/time.py).
This notebook just stitches them together.

**Per CLAUDE.md**:
- Time-aware split via `temporal_split`; NEVER `train_test_split(shuffle=True)`.
- `class_weight='balanced'` for class imbalance; NEVER SMOTE.
- Headline metrics: **PR-AUC** + **precision@top-decile**, plus calibration.

**Deliverables**
- `data/models/baseline_<YYYYMMDD>.joblib` — trained calibrated estimator
- `data/models/baseline_<YYYYMMDD>_metadata.json` — split cutoffs, features, metrics
- `reports/metrics/baseline_<YYYYMMDD>.json` — same metrics, checked into git for diffing across runs

## 1. Setup

In [ ]:
import sys
import json
from datetime import date
from pathlib import Path

# Notebook lives in notebooks/; package lives in src/foodsafety/. Add ../src.
# In Jupyter/VS Code notebooks, __file__ is not defined, so use CWD.
# Assumes notebook is run from project root (or cwd is set to project root).
_PROJECT_ROOT = Path.cwd() if Path.cwd().name != 'notebooks' else Path.cwd().parent
if str(_PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / 'src'))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV

## 2. Load + define splits

Cutoffs:
- **train**: 2019-01-01 → 2024-06-30 (~5.5 years, the bulk of trainable history)
- **val**: 2024-07-01 → 2025-06-30 (one year, used for isotonic calibration
  and any threshold tuning)
- **test**: 2025-07-01 → dataset_max (the most recent slice — closest to the
  conditions we'd deploy under)

Per CLAUDE.md these cutoffs are deliberate: NEVER
`train_test_split(shuffle=True)` and never cross-validate across the train
boundary. The splitter is the only allowed split helper.

In [ ]:
features_path = PROCESSED_DIR / 'features.parquet'
if not features_path.exists():
    raise SystemExit(
        f'Missing {features_path}. Run notebooks 02 + 03 first.'
    )

features = pd.read_parquet(features_path)
features['inspection_date'] = pd.to_datetime(features['inspection_date'])
# Cast bool flag columns to int8 — XGBoost / sklearn handle int faster than bool.
for c in features.columns:
    if c.startswith('flag_kw_'):
        features[c] = features[c].astype('int8')

print(f'features: {len(features):,} rows × {features.shape[1]} cols')
print(f'date range: {features["inspection_date"].min().date()} → {features["inspection_date"].max().date()}')

TRAIN_END = '2024-07-01'
VAL_END   = '2025-07-01'

split = temporal_split(features, train_end=TRAIN_END, val_end=VAL_END)

for name, frame in [('train', split.train), ('val', split.val), ('test', split.test)]:
    s = summarize(frame, label_col=LABEL_COL)
    print(f'  {name:<5}  n={s.rows:>6,}  '
          f'dates {s.date_min.date()} → {s.date_max.date()}  '
          f'positive_rate={s.positive_rate:.2%}')

## 3. Fit

Two-step training:
1. **Fit** the pipeline (preprocessor + LogReg with `class_weight='balanced'`)
   on the train set.
2. **Calibrate** the fitted estimator against the val set using isotonic
   regression. We use `CalibratedClassifierCV(cv='prefit')` so the fit step
   above isn't re-done; the calibrator just learns predicted-→-empirical
   on the val predictions.

Class-weight balancing reweights the loss to address the ~13% positive rate.
Per CLAUDE.md this is the mandated alternative to SMOTE.

In [ ]:
%%time
X_train = split.train[ALL_FEATURES]
y_train = split.train[LABEL_COL].astype(int)
X_val   = split.val  [ALL_FEATURES]
y_val   = split.val  [LABEL_COL].astype(int)
X_test  = split.test [ALL_FEATURES]
y_test  = split.test [LABEL_COL].astype(int)

print(f'Train shape: X={X_train.shape}, y positive rate={y_train.mean():.2%}')

# Step 1 — fit the uncalibrated pipeline
pipe = build_baseline_pipeline(random_state=RANDOM_STATE)
pipe.fit(X_train, y_train)

# Step 2 — fit an isotonic calibrator on val (prefit so the inner estimator
# isn't refit; only the calibration mapping is learned).
model = CalibratedClassifierCV(pipe, method='isotonic', cv='prefit')
model.fit(X_val, y_val)

print('\nfeature space after preprocessing:')
n_out = pipe.named_steps['preprocess'].transform(X_train.head(10)).shape[1]
print(f'  {n_out} columns out of preprocess (after one-hot expansion)')

## 4. Evaluate on val and test

Two reports: val (the data the calibrator was fit on — performance is
optimistic) and test (untouched until now — this is what we report).

Headline numbers we care about, in order:
  1. **PR-AUC** — primary metric, robust to class imbalance
  2. **Precision@10%** — operational metric: of the top decile we flag, what fraction are positive?
  3. **Top-decile lift** — should be ≥ 2× for a useful model
  4. **Calibration** — does a 0.30 score actually mean ~30% empirical rate?
  5. ROC-AUC — reported but not used as the decision metric

In [ ]:
val_scores  = model.predict_proba(X_val)[:, 1]
test_scores = model.predict_proba(X_test)[:, 1]

val_report  = evaluate(y_val,  val_scores)
test_report = evaluate(y_test, test_scores)

summary = pd.DataFrame({
    'val':  val_report.to_dict(),
    'test': test_report.to_dict(),
})
print(summary.to_string())

### 4a. Decile lift table (TEST set)

Decile 1 = top 10% of predicted scores. A well-ranked model has decile-1 lift > 2.

In [ ]:
lift = decile_lift_table(y_test, test_scores)
print('Decile lift (TEST):')
print(lift.to_string())

### 4b. Calibration (TEST set)

If isotonic calibration worked, `mean_observed` should track `mean_predicted` across bins.

In [ ]:
calib = calibration_table(y_test, test_scores, n_bins=10)
print(calib.to_string())

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color='#9CA3AF', linestyle=':', label='Perfect')
ax.plot(calib['mean_predicted'], calib['mean_observed'], 'o-', color='#15110D',
        label='Baseline (test)')
ax.set_xlabel('Mean predicted'); ax.set_ylabel('Mean observed')
ax.set_title('Calibration curve — test set'); ax.legend()
ax.set_xlim(0, max(calib['mean_predicted'].max(), calib['mean_observed'].max()) * 1.1)
ax.set_ylim(0, max(calib['mean_predicted'].max(), calib['mean_observed'].max()) * 1.1)
plt.tight_layout(); plt.show()

## 5. Persist model + metadata

Per CLAUDE.md: never overwrite a model file; include date in the filename;
always write a metadata sidecar with train cutoff, features, and metrics.

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS_DIR = _PROJECT_ROOT / 'reports' / 'metrics'
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)

stamp = date.today().isoformat().replace('-', '')
model_path = MODELS_DIR / f'baseline_{stamp}.joblib'
metadata_path = MODELS_DIR / f'baseline_{stamp}_metadata.json'
report_path = REPORTS_METRICS_DIR / f'baseline_{stamp}.json'

joblib.dump(model, model_path)
print(f'saved model → {model_path}  ({model_path.stat().st_size / 1e6:.1f} MB)')

metadata = {
    'model': 'baseline_logreg_isotonic',
    'random_state': RANDOM_STATE,
    'date_trained': date.today().isoformat(),
    'split': {
        'train_end': str(split.train_end.date()),
        'val_end':   str(split.val_end.date()),
        'train_n':   int(len(split.train)),
        'val_n':     int(len(split.val)),
        'test_n':    int(len(split.test)),
    },
    'features': {
        'all':         ALL_FEATURES,
        'label_col':   LABEL_COL,
    },
    'metrics': {
        'val':  val_report.to_dict(),
        'test': test_report.to_dict(),
    },
    'features_parquet_mtime': pd.Timestamp(features_path.stat().st_mtime, unit='s').isoformat(),
}
with metadata_path.open('w') as f:
    json.dump(metadata, f, indent=2)
print(f'saved metadata → {metadata_path}')

# Same metrics duplicated to reports/metrics so they're git-tracked + diffable
with report_path.open('w') as f:
    json.dump({'model': 'baseline', 'val': val_report.to_dict(), 'test': test_report.to_dict()}, f, indent=2)
print(f'saved report → {report_path}')

## 6. Hand-off

**Baseline performance on TEST**:
Headline numbers printed in § 4. Pin them in the model card. Phase 5 (XGBoost)
must clear the baseline on **both** PR-AUC and precision@10% before we
consider it the production estimator.

**Next step**: `notebooks/05_xgboost_model.ipynb` — XGBoost with the same
feature contract (imported from `baseline.py`), same split (same cutoffs),
and the same eval suite. Direct comparison.

**Sanity to confirm before moving on**:
- Test PR-AUC > val positive rate (otherwise model is worse than the no-skill predictor)
- Top-decile lift ≥ 1.5×
- Calibration curve roughly on the diagonal
- `tests/` still all green